# Lista de Exercícios - Variáveis Dummies, Modelos Logit e Probit

**Disciplina:** Introdução à Econometria  
**Professor:** Cássio da Nóbrega Besarria  
**Aluno:** Pedro Rebouças Veloso

---

## Base de Dados Escolhida: Adult Census Income

**Fonte:** UCI Machine Learning Repository  
**Objetivo:** Prever se um indivíduo possui renda anual superior a $50.000 (variável binária).

**Justificativa da escolha:** Esta base é amplamente utilizada em estudos econométricos por permitir análise de fatores socioeconômicos que influenciam a probabilidade de alta renda. Possui variáveis quantitativas (idade, horas trabalhadas) e qualitativas (sexo, educação, ocupação), sendo ideal para aplicação de variáveis dummies e modelos de resposta binária.

In [1]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit, Probit
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Configuração de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

---
## 1. Importação da Base de Dados

In [2]:
# Definição dos nomes das colunas conforme documentação do UCI
colunas = ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 
           'marital_status', 'occupation', 'relationship', 'race', 'sex',
           'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']

# Carregamento direto do repositório UCI
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
df = pd.read_csv(url, names=colunas, na_values='?', skipinitialspace=True)

# Remoção de valores ausentes para simplificar a análise
# Nota: dataset original tem 32.561 obs; após remoção de ausentes, fica com ~30.162
df = df.dropna()

print(f'a) Número de observações: {len(df)}')
print(f'   Número de variáveis: {len(df.columns)}')

a) Número de observações: 30162
   Número de variáveis: 15


In [3]:
# b) Descrição das variáveis
print('b) Descrição das variáveis:\n')
descricao = {
    'age': 'Idade do indivíduo (contínua)',
    'workclass': 'Tipo de empregador (categórica)',
    'fnlwgt': 'Peso amostral - será removida (não relevante para análise)',
    'education': 'Nível de escolaridade (categórica)',
    'education_num': 'Anos de estudo (contínua)',
    'marital_status': 'Estado civil (categórica)',
    'occupation': 'Ocupação profissional (categórica)',
    'relationship': 'Relação familiar (categórica)',
    'race': 'Raça/etnia (categórica)',
    'sex': 'Sexo (categórica binária)',
    'capital_gain': 'Ganhos de capital (contínua)',
    'capital_loss': 'Perdas de capital (contínua)',
    'hours_per_week': 'Horas trabalhadas por semana (contínua)',
    'native_country': 'País de origem (categórica)',
    'income': 'Renda >50K ou <=50K (variável dependente binária)'
}

for var, desc in descricao.items():
    print(f'  - {var}: {desc}')

b) Descrição das variáveis:

  - age: Idade do indivíduo (contínua)
  - workclass: Tipo de empregador (categórica)
  - fnlwgt: Peso amostral - será removida (não relevante para análise)
  - education: Nível de escolaridade (categórica)
  - education_num: Anos de estudo (contínua)
  - marital_status: Estado civil (categórica)
  - occupation: Ocupação profissional (categórica)
  - relationship: Relação familiar (categórica)
  - race: Raça/etnia (categórica)
  - sex: Sexo (categórica binária)
  - capital_gain: Ganhos de capital (contínua)
  - capital_loss: Perdas de capital (contínua)
  - hours_per_week: Horas trabalhadas por semana (contínua)
  - native_country: País de origem (categórica)
  - income: Renda >50K ou <=50K (variável dependente binária)


In [4]:
# c) Identificação da variável dependente e explicativas
print('c) Identificação das variáveis:')
print('VARIÁVEL DEPENDENTE (Y):')
print('  - income: indica se o indivíduo ganha mais de $50.000/ano (1) ou não (0)')

print('VARIÁVEIS EXPLICATIVAS (X):')
print('  Quantitativas:')
print('    - age: captura efeito da experiência sobre renda')
print('    - education_num: proxy para capital humano')
print('    - hours_per_week: intensidade do trabalho')

print('  Qualitativas (serão transformadas em dummies):')
print('    - sex: diferencial de gênero no mercado de trabalho')
print('    - marital_status: efeito do estado civil')
print('    - education: nível educacional categórico')

c) Identificação das variáveis:
VARIÁVEL DEPENDENTE (Y):
  - income: indica se o indivíduo ganha mais de $50.000/ano (1) ou não (0)
VARIÁVEIS EXPLICATIVAS (X):
  Quantitativas:
    - age: captura efeito da experiência sobre renda
    - education_num: proxy para capital humano
    - hours_per_week: intensidade do trabalho
  Qualitativas (serão transformadas em dummies):
    - sex: diferencial de gênero no mercado de trabalho
    - marital_status: efeito do estado civil
    - education: nível educacional categórico


In [5]:
# d) Justificativa para a escolha da variável dependente binária
print('d) Justificativa da variável dependente binária:\n')
print('''A variável "income" é adequada como dependente binária porque:

1. NATUREZA BINÁRIA: Assume apenas dois valores (>50K ou <=50K), 
   atendendo ao requisito dos modelos Logit e Probit.

2. RELEVÂNCIA ECONÔMICA: O limiar de $50.000 representa um ponto de corte
   significativo que separa classes de renda média e alta nos EUA.

3. INTERESSE ANALÍTICO: Permite investigar quais fatores aumentam a
   probabilidade de um indivíduo pertencer ao grupo de alta renda.

4. APLICABILIDADE: É possível derivar implicações de política pública
   (ex: retorno da educação, discriminação salarial por gênero).''')

d) Justificativa da variável dependente binária:

A variável "income" é adequada como dependente binária porque:

1. NATUREZA BINÁRIA: Assume apenas dois valores (>50K ou <=50K), 
   atendendo ao requisito dos modelos Logit e Probit.

2. RELEVÂNCIA ECONÔMICA: O limiar de $50.000 representa um ponto de corte
   significativo que separa classes de renda média e alta nos EUA.

3. INTERESSE ANALÍTICO: Permite investigar quais fatores aumentam a
   probabilidade de um indivíduo pertencer ao grupo de alta renda.

4. APLICABILIDADE: É possível derivar implicações de política pública
   (ex: retorno da educação, discriminação salarial por gênero).


In [6]:
# Criação da variável dependente binária
# 1 = renda > 50K, 0 = renda <= 50K
df['income_binary'] = (df['income'] == '>50K').astype(int)

# Verificação
print('Conversão da variável dependente:')
print(df[['income', 'income_binary']].drop_duplicates())

Conversão da variável dependente:
  income  income_binary
0  <=50K              0
7   >50K              1


---
## 2. Análise Descritiva da Base de Dados

In [7]:
# a) Estatísticas descritivas das variáveis quantitativas
print('a) Estatísticas descritivas das variáveis quantitativas:\n')

vars_quantitativas = ['age', 'education_num', 'hours_per_week', 'capital_gain', 'capital_loss']
estatisticas = df[vars_quantitativas].describe().T
estatisticas['cv'] = estatisticas['std'] / estatisticas['mean']  # Coeficiente de variação
print(estatisticas[['count', 'mean', 'std', 'min', '50%', 'max', 'cv']])

a) Estatísticas descritivas das variáveis quantitativas:

                    count      mean       std     min     50%        max  \
age            30162.0000   38.4379   13.1347 17.0000 37.0000    90.0000   
education_num  30162.0000   10.1213    2.5500  1.0000 10.0000    16.0000   
hours_per_week 30162.0000   40.9312   11.9800  1.0000 40.0000    99.0000   
capital_gain   30162.0000 1092.0079 7406.3465  0.0000  0.0000 99999.0000   
capital_loss   30162.0000   88.3725  404.2984  0.0000  0.0000  4356.0000   

                   cv  
age            0.3417  
education_num  0.2519  
hours_per_week 0.2927  
capital_gain   6.7823  
capital_loss   4.5749  


In [8]:
# b) Frequência da variável dependente
print('b) Frequência da variável dependente (income):\n')

freq_income = df['income_binary'].value_counts()
freq_income_pct = df['income_binary'].value_counts(normalize=True) * 100

tabela_freq = pd.DataFrame({
    'Frequência': freq_income,
    'Percentual (%)': freq_income_pct
})
tabela_freq.index = ['Renda <= 50K (0)', 'Renda > 50K (1)']
print(tabela_freq)
print(f'\nTotal: {len(df)} observações')

b) Frequência da variável dependente (income):

                  Frequência  Percentual (%)
Renda <= 50K (0)       22654         75.1078
Renda > 50K (1)         7508         24.8922

Total: 30162 observações


In [9]:
# c) Frequência das variáveis qualitativas
print('c) Frequência das variáveis qualitativas:\n')

# Sexo
print('SEXO:')
print(df['sex'].value_counts())
print()

# Estado civil (agrupado para simplificar)
print('ESTADO CIVIL:')
print(df['marital_status'].value_counts())
print()

# Educação
print('NÍVEL EDUCACIONAL (top 5):')
print(df['education'].value_counts().head())

c) Frequência das variáveis qualitativas:

SEXO:
sex
Male      20380
Female     9782
Name: count, dtype: int64

ESTADO CIVIL:
marital_status
Married-civ-spouse       14065
Never-married             9726
Divorced                  4214
Separated                  939
Widowed                    827
Married-spouse-absent      370
Married-AF-spouse           21
Name: count, dtype: int64

NÍVEL EDUCACIONAL (top 5):
education
HS-grad         9840
Some-college    6678
Bachelors       5044
Masters         1627
Assoc-voc       1307
Name: count, dtype: int64


In [10]:
# d) Interpretação econômica dos resultados
print('d) Interpretação econômica dos resultados descritivos:\n')

# Calculando algumas estatísticas para interpretação
pct_alta_renda = (df['income_binary'].mean() * 100)
media_idade = df['age'].mean()
media_horas = df['hours_per_week'].mean()
media_educ = df['education_num'].mean()
pct_masculino = (df['sex'] == 'Male').mean() * 100

# Taxa de alta renda por sexo
taxa_masc = df[df['sex'] == 'Male']['income_binary'].mean() * 100
taxa_fem = df[df['sex'] == 'Female']['income_binary'].mean() * 100


d) Interpretação econômica dos resultados descritivos:



INTERPRETAÇÃO ECONÔMICA:

1. DISTRIBUIÇÃO DE RENDA:
   - Apenas {pct_alta_renda:.1f}% da amostra possui renda superior a $50K.
   - Isso indica uma distribuição assimétrica típica de renda.

2. PERFIL DA AMOSTRA:
   - Idade média de {media_idade:.1f} anos (população economicamente ativa).
   - Média de {media_horas:.1f} horas trabalhadas por semana.
   - Escolaridade média de {media_educ:.1f} anos de estudo.

3. DIFERENCIAL POR GÊNERO:
   - Homens: {taxa_masc:.1f}% com alta renda
   - Mulheres: {taxa_fem:.1f}% com alta renda
   - Existe evidência preliminar de diferencial salarial por gênero,
     que será investigado nos modelos econométricos.

4. CAPITAL HUMANO:
   - A variável education_num apresenta dispersão considerável,
     permitindo capturar o efeito da escolaridade sobre a renda.''')

---
## 3. Fundamentação Teórica: Variáveis Dummies


a) O QUE SÃO VARIÁVEIS DUMMIES?

Variáveis dummies (ou indicadoras) são variáveis binárias que assumem valores
0 ou 1 para representar a presença ou ausência de uma característica qualitativa.

Exemplo: Para a variável "sexo":
  - D_masculino = 1 se o indivíduo é homem, 0 caso contrário

Matematicamente, se temos uma variável categórica com k categorias,
criamos (k-1) variáveis dummies para evitar multicolinearidade perfeita.

---

b) FINALIDADE EM MODELOS ECONOMÉTRICOS:

1. QUANTIFICAR EFEITOS QUALITATIVOS: Permitem incluir fatores não-numéricos
   (sexo, região, estado civil) em modelos de regressão.

2. CAPTURAR DIFERENÇAS ENTRE GRUPOS: O coeficiente da dummy mede a diferença
   média no Y entre a categoria representada e a categoria de referência.

3. FLEXIBILIDADE: Podem capturar efeitos não-lineares e interações entre
   variáveis qualitativas e quantitativas.

---

c) CATEGORIA DE REFERÊNCIA:

É a categoria omitida na criação das dummies. Seus efeitos são absorvidos
pelo intercepto (constante) do modelo.

Os coeficientes das dummies representam a diferença em relação a esta categoria.

Exemplo: Se "Feminino" é a referência e β_masculino = 0.15, significa que
homens têm, em média, probabilidade 15 p.p. maior de alta renda que mulheres.

---

d) ARMADILHA DAS VARIÁVEIS DUMMIES (Dummy Variable Trap):

Ocorre quando incluímos todas as k dummies de uma variável categórica com k
categorias, junto com o intercepto. Isso gera MULTICOLINEARIDADE PERFEITA:

  D1 + D2 + ... + Dk = 1 (sempre)

O sistema se torna indeterminado (matriz X'X singular, não-invertível).

SOLUÇÃO: Sempre omitir uma categoria (usar k-1 dummies) ou remover o intercepto.
A prática padrão é manter o intercepto e usar (k-1) dummies.
''')

---
## 4. Construção das Variáveis Dummies

In [11]:
# a) Procedimento utilizado
print('a) Procedimento para construção das dummies:\n')
print('''Utilizaremos pd.get_dummies() com drop_first=True para:
1. Converter automaticamente variáveis categóricas em dummies
2. Omitir a primeira categoria (evitando a armadilha das dummies)
''')

# Seleção das variáveis para o modelo
# Justificativa: escolhemos variáveis com respaldo na teoria econômica
# - age: experiência (teoria do capital humano)
# - education_num: escolaridade (teoria do capital humano)
# - hours_per_week: oferta de trabalho
# - sex: discriminação no mercado de trabalho
# - marital_status: efeitos de seleção e produtividade

# Criando dummy para sexo manualmente para maior controle
df['male'] = (df['sex'] == 'Male').astype(int)

# Simplificando estado civil em casado vs não-casado
# Justificativa: literatura indica que casados têm prêmio salarial
df['married'] = df['marital_status'].isin(['Married-civ-spouse', 'Married-AF-spouse']).astype(int)

# Criando dummy para ensino superior (educação >= 13 anos)
# Justificativa: capturar o efeito do diploma além dos anos de estudo
# Nota: incluímos education_num E college para isolar o salto do diploma
df['college'] = (df['education_num'] >= 13).astype(int)

print('Dummies criadas com sucesso!')

a) Procedimento para construção das dummies:

Utilizaremos pd.get_dummies() com drop_first=True para:
1. Converter automaticamente variáveis categóricas em dummies
2. Omitir a primeira categoria (evitando a armadilha das dummies)

Dummies criadas com sucesso!


In [12]:
# b) Categoria de referência adotada
print('b) Categorias de referência:\n')
print('''Para cada variável dummy criada:

  - male: Categoria de referência = Feminino (Female)
    Interpretação: coeficiente mede diferença homens vs mulheres

  - married: Categoria de referência = Não-casado
    Interpretação: coeficiente mede diferença casados vs não-casados

  - college: Categoria de referência = Sem ensino superior
    Interpretação: coeficiente mede efeito do diploma universitário

A escolha das referências segue a convenção de usar a categoria
teoricamente associada a menor renda como base de comparação.''')

b) Categorias de referência:

Para cada variável dummy criada:

  - male: Categoria de referência = Feminino (Female)
    Interpretação: coeficiente mede diferença homens vs mulheres

  - married: Categoria de referência = Não-casado
    Interpretação: coeficiente mede diferença casados vs não-casados

  - college: Categoria de referência = Sem ensino superior
    Interpretação: coeficiente mede efeito do diploma universitário

A escolha das referências segue a convenção de usar a categoria
teoricamente associada a menor renda como base de comparação.


In [13]:
# c) Interpretação econômica das variáveis criadas
print('c) Interpretação econômica das variáveis dummies:\n')

# Tabela cruzada para verificar relação com Y
print('Taxa de alta renda por grupo:\n')

for var in ['male', 'married', 'college']:
    taxa = df.groupby(var)['income_binary'].mean() * 100
    print(f'{var.upper()}:')
    print(f'  {var}=0: {taxa[0]:.1f}% com renda >50K')
    print(f'  {var}=1: {taxa[1]:.1f}% com renda >50K')
    print(f'  Diferença: {taxa[1] - taxa[0]:.1f} pontos percentuais\n')

print('''INTERPRETAÇÃO:
- Homens têm probabilidade substancialmente maior de alta renda que mulheres.
- Casados apresentam maior probabilidade de alta renda (prêmio do casamento).
- Diploma universitário está fortemente associado a maior renda.

Estas diferenças brutas serão refinadas nos modelos multivariados.''')

c) Interpretação econômica das variáveis dummies:

Taxa de alta renda por grupo:

MALE:
  male=0: 11.4% com renda >50K
  male=1: 31.4% com renda >50K
  Diferença: 20.0 pontos percentuais

MARRIED:
  married=0: 6.8% com renda >50K
  married=1: 45.5% com renda >50K
  Diferença: 38.7 pontos percentuais

COLLEGE:
  college=0: 16.7% com renda >50K
  college=1: 49.2% com renda >50K
  Diferença: 32.4 pontos percentuais

INTERPRETAÇÃO:
- Homens têm probabilidade substancialmente maior de alta renda que mulheres.
- Casados apresentam maior probabilidade de alta renda (prêmio do casamento).
- Diploma universitário está fortemente associado a maior renda.

Estas diferenças brutas serão refinadas nos modelos multivariados.


---
## 5. Modelo de Probabilidade Linear (MQO)

In [14]:
# Preparação dos dados para estimação
# Variáveis explicativas selecionadas com base na teoria econômica
X = df[['age', 'education_num', 'hours_per_week', 'male', 'married', 'college']]
X = sm.add_constant(X)  # Adiciona intercepto
y = df['income_binary']

# a) Estimação do Modelo de Probabilidade Linear (OLS)
# Nota: usamos erros-padrão robustos (HC1) devido à heterocedasticidade inerente
modelo_mpl = sm.OLS(y, X).fit(cov_type='HC1')

print('a) Modelo de Probabilidade Linear - Equação Estimada:')
print('P(Y=1|X) = β₀ + β₁*age + β₂*education_num + β₃*hours_per_week + β₄*male + β₅*married + β₆*college + ε')
print('(Erros-padrão robustos à heterocedasticidade - HC1)')
print(modelo_mpl.summary().tables[1])

a) Modelo de Probabilidade Linear - Equação Estimada:
P(Y=1|X) = β₀ + β₁*age + β₂*education_num + β₃*hours_per_week + β₄*male + β₅*married + β₆*college + ε
(Erros-padrão robustos à heterocedasticidade - HC1)
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -0.5463      0.013    -42.353      0.000      -0.572      -0.521
age                0.0034      0.000     20.792      0.000       0.003       0.004
education_num      0.0345      0.001     31.535      0.000       0.032       0.037
hours_per_week     0.0031      0.000     16.705      0.000       0.003       0.003
male               0.0247      0.004      5.613      0.000       0.016       0.033
married            0.3087      0.005     58.958      0.000       0.298       0.319
college            0.1104      0.008     14.435      0.000       0.095       0.125


b) Interpretação econômica dos coeficientes (MPL)

In [15]:

coefs = modelo_mpl.params

print(f'''No Modelo de Probabilidade Linear, os coeficientes representam
a variação na PROBABILIDADE de Y=1 dado uma variação unitária em X.

COEFICIENTES ESTIMADOS:

• Intercepto ({coefs['const']:.4f}): Probabilidade base quando todas as
  variáveis são zero (sem interpretação prática direta).

• Age ({coefs['age']:.4f}): Cada ano adicional de idade aumenta a
  probabilidade de alta renda em {coefs['age']*100:.2f} pontos percentuais.
  Reflete acúmulo de experiência no mercado de trabalho.

• Education_num ({coefs['education_num']:.4f}): Cada ano adicional de
  escolaridade aumenta a probabilidade em {coefs['education_num']*100:.2f} p.p.
  Evidência do retorno da educação (teoria do capital humano).

• Hours_per_week ({coefs['hours_per_week']:.4f}): Cada hora adicional
  trabalhada aumenta a probabilidade em {coefs['hours_per_week']*100:.2f} p.p.

• Male ({coefs['male']:.4f}): Ser homem aumenta a probabilidade de alta
  renda em {coefs['male']*100:.1f} p.p. comparado a mulheres, ceteris paribus.
  Evidência de diferencial de gênero no mercado de trabalho.

• Married ({coefs['married']:.4f}): Ser casado aumenta a probabilidade
  em {coefs['married']*100:.1f} p.p. ("prêmio do casamento").

• College ({coefs['college']:.4f}): Possuir diploma universitário aumenta
  a probabilidade em {coefs['college']*100:.1f} p.p. (efeito do diploma).''')

No Modelo de Probabilidade Linear, os coeficientes representam
a variação na PROBABILIDADE de Y=1 dado uma variação unitária em X.

COEFICIENTES ESTIMADOS:

• Intercepto (-0.5463): Probabilidade base quando todas as
  variáveis são zero (sem interpretação prática direta).

• Age (0.0034): Cada ano adicional de idade aumenta a
  probabilidade de alta renda em 0.34 pontos percentuais.
  Reflete acúmulo de experiência no mercado de trabalho.

• Education_num (0.0345): Cada ano adicional de
  escolaridade aumenta a probabilidade em 3.45 p.p.
  Evidência do retorno da educação (teoria do capital humano).

• Hours_per_week (0.0031): Cada hora adicional
  trabalhada aumenta a probabilidade em 0.31 p.p.

• Male (0.0247): Ser homem aumenta a probabilidade de alta
  renda em 2.5 p.p. comparado a mulheres, ceteris paribus.
  Evidência de diferencial de gênero no mercado de trabalho.

• Married (0.3087): Ser casado aumenta a probabilidade
  em 30.9 p.p. ("prêmio do casamento").

• College (0.1104)

In [16]:
# c) Coeficiente de determinação R²
print('c) Coeficiente de Determinação:\n')
print(f'R² = {modelo_mpl.rsquared:.4f}')
print(f'R² ajustado = {modelo_mpl.rsquared_adj:.4f}')
print(f'''\nInterpretação: O modelo explica aproximadamente {modelo_mpl.rsquared*100:.1f}% da
variação na probabilidade de alta renda. Embora pareça baixo, é comum em
modelos de escolha discreta, onde há muita variabilidade individual não
capturada pelas variáveis observáveis.''')

c) Coeficiente de Determinação:

R² = 0.3136
R² ajustado = 0.3134

Interpretação: O modelo explica aproximadamente 31.4% da
variação na probabilidade de alta renda. Embora pareça baixo, é comum em
modelos de escolha discreta, onde há muita variabilidade individual não
capturada pelas variáveis observáveis.


In [17]:
# d) Limitações do modelo
print('d) Limitações do Modelo de Probabilidade Linear:\n')

# Verificando probabilidades estimadas fora do intervalo [0,1]
y_pred_mpl = modelo_mpl.predict(X)
fora_intervalo = ((y_pred_mpl < 0) | (y_pred_mpl > 1)).sum()
pct_fora = fora_intervalo / len(y_pred_mpl) * 100

print(f'''LIMITAÇÕES DO MPL:

1. PROBABILIDADES FORA DO INTERVALO [0,1]:
   - {fora_intervalo} observações ({pct_fora:.2f}%) com probabilidade < 0 ou > 1
   - Probabilidade mínima prevista: {y_pred_mpl.min():.4f}
   - Probabilidade máxima prevista: {y_pred_mpl.max():.4f}
   - Isso viola a definição de probabilidade!

2. HETEROCEDASTICIDADE INERENTE:
   - Var(Y|X) = P(1-P), que varia com X
   - Viola a hipótese de homocedasticidade do MQO
   - Por isso usamos erros-padrão robustos (HC1) na estimação acima

3. EFEITOS MARGINAIS CONSTANTES:
   - Assume que o efeito de X sobre P(Y=1) é constante
   - Na realidade, efeitos podem ser maiores/menores dependendo do nível de P

4. DISTRIBUIÇÃO DOS ERROS:
   - Erros não são normalmente distribuídos (Y é binário)
   - Compromete inferência em amostras pequenas

=> Por estas razões, os modelos Logit e Probit são preferíveis.''')

d) Limitações do Modelo de Probabilidade Linear:

LIMITAÇÕES DO MPL:

1. PROBABILIDADES FORA DO INTERVALO [0,1]:
   - 5293 observações (17.55%) com probabilidade < 0 ou > 1
   - Probabilidade mínima prevista: -0.3874
   - Probabilidade máxima prevista: 0.9431
   - Isso viola a definição de probabilidade!

2. HETEROCEDASTICIDADE INERENTE:
   - Var(Y|X) = P(1-P), que varia com X
   - Viola a hipótese de homocedasticidade do MQO
   - Por isso usamos erros-padrão robustos (HC1) na estimação acima

3. EFEITOS MARGINAIS CONSTANTES:
   - Assume que o efeito de X sobre P(Y=1) é constante
   - Na realidade, efeitos podem ser maiores/menores dependendo do nível de P

4. DISTRIBUIÇÃO DOS ERROS:
   - Erros não são normalmente distribuídos (Y é binário)
   - Compromete inferência em amostras pequenas

=> Por estas razões, os modelos Logit e Probit são preferíveis.


---
## 6. Fundamentação Teórica: Modelos Logit e Probit


a) POR QUE O MQO NÃO É ADEQUADO PARA VARIÁVEL DEPENDENTE BINÁRIA?

O MQO modela E(Y|X) = Xβ linearmente, mas quando Y ∈ {0,1}:

1. E(Y|X) = P(Y=1|X) deve estar entre 0 e 1
2. O modelo linear não respeita esta restrição
3. Gera heterocedasticidade intrínseca: Var(Y|X) = P(1-P)
4. Assume efeitos marginais constantes (não realista)

SOLUÇÃO: Usar uma função de ligação que mapeie Xβ ∈ (-∞, +∞) para P ∈ (0,1)

---

b) DIFERENÇAS ENTRE OS MODELOS LOGIT E PROBIT:

Ambos modelam: P(Y=1|X) = F(Xβ), onde F é uma função de distribuição acumulada.

LOGIT:
  - Usa a função logística: F(z) = exp(z) / [1 + exp(z)] = 1 / [1 + exp(-z)]
  - Baseada na distribuição logística
  - Caudas mais pesadas (maior probabilidade em extremos)
  - Permite interpretação via razão de chances (odds ratio)

PROBIT:
  - Usa a função de distribuição normal acumulada: F(z) = Φ(z)
  - Baseada na distribuição normal
  - Caudas mais leves
  - Derivado do modelo de variável latente com erro normal

SEMELHANÇA: As curvas são muito similares. Os coeficientes do Logit são
aproximadamente 1.6 vezes os do Probit (devido às diferentes variâncias).

---

c) VANTAGENS E LIMITAÇÕES:

LOGIT:
  Vantagens:
    - Interpretação via odds ratio: exp(β) é a razão de chances
    - Computacionalmente mais simples
    - Mais comum em ciências sociais e epidemiologia
  Limitações:
    - Caudas mais pesadas podem superestimar probabilidades extremas

PROBIT:
  Vantagens:
    - Fundamentação teórica via variável latente com erro normal
    - Mais comum em economia (modelos estruturais)
    - Fundamentação via modelo de variável latente com erro normal
  Limitações:
    - Coeficientes não têm interpretação direta
    - Requer cálculo de efeitos marginais para interpretação

AMBOS:
  - Estimação por Máxima Verossimilhança (não MQO)
  - Requerem amostras maiores para boas propriedades
  - Coeficientes não representam diretamente efeitos marginais
''')

---
## 7. Estimação do Modelo Logit

In [18]:
# a) Estimação do modelo Logit
modelo_logit = Logit(y, X).fit(disp=0)  # disp=0 suprime output de convergência

print('a) Modelo Logit - Equação Estimada:\n')
print('P(Y=1|X) = Λ(β₀ + β₁*age + β₂*education_num + β₃*hours_per_week + β₄*male + β₅*married + β₆*college)')
print('\nOnde Λ(z) = exp(z)/[1+exp(z)] é a função logística.\n')
print(modelo_logit.summary().tables[1])

a) Modelo Logit - Equação Estimada:

P(Y=1|X) = Λ(β₀ + β₁*age + β₂*education_num + β₃*hours_per_week + β₄*male + β₅*married + β₆*college)

Onde Λ(z) = exp(z)/[1+exp(z)] é a função logística.

                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -9.1235      0.167    -54.471      0.000      -9.452      -8.795
age                0.0332      0.001     23.536      0.000       0.030       0.036
education_num      0.3543      0.014     26.032      0.000       0.328       0.381
hours_per_week     0.0311      0.001     20.798      0.000       0.028       0.034
male               0.1233      0.046      2.670      0.008       0.033       0.214
married            2.2922      0.043     52.755      0.000       2.207       2.377
college            0.1798      0.065      2.757      0.006       0.052       0.308


In [19]:
# b) Interpretação dos sinais dos coeficientes
print('b) Interpretação dos sinais dos coeficientes:\n')

coefs_logit = modelo_logit.params

print('''No modelo Logit, os SINAIS dos coeficientes indicam a DIREÇÃO do efeito:
  - Coeficiente positivo: aumenta a probabilidade de Y=1
  - Coeficiente negativo: diminui a probabilidade de Y=1

ANÁLISE DOS SINAIS:\n''')

for var in coefs_logit.index:
    sinal = '+' if coefs_logit[var] > 0 else '-'
    efeito = 'AUMENTA' if coefs_logit[var] > 0 else 'DIMINUI'
    print(f'  {var}: {sinal} ({efeito} probabilidade de alta renda)')

print('''\nTodos os coeficientes são positivos, indicando que:
- Maior idade, escolaridade e horas trabalhadas aumentam P(renda>50K)
- Ser homem, casado e ter diploma universitário aumentam P(renda>50K)

Os resultados são consistentes com a teoria econômica do capital humano
e com a literatura sobre diferenciais salariais.''')

b) Interpretação dos sinais dos coeficientes:

No modelo Logit, os SINAIS dos coeficientes indicam a DIREÇÃO do efeito:
  - Coeficiente positivo: aumenta a probabilidade de Y=1
  - Coeficiente negativo: diminui a probabilidade de Y=1

ANÁLISE DOS SINAIS:

  const: - (DIMINUI probabilidade de alta renda)
  age: + (AUMENTA probabilidade de alta renda)
  education_num: + (AUMENTA probabilidade de alta renda)
  hours_per_week: + (AUMENTA probabilidade de alta renda)
  male: + (AUMENTA probabilidade de alta renda)
  married: + (AUMENTA probabilidade de alta renda)
  college: + (AUMENTA probabilidade de alta renda)

Todos os coeficientes são positivos, indicando que:
- Maior idade, escolaridade e horas trabalhadas aumentam P(renda>50K)
- Ser homem, casado e ter diploma universitário aumentam P(renda>50K)

Os resultados são consistentes com a teoria econômica do capital humano
e com a literatura sobre diferenciais salariais.


In [20]:
# c) Significância estatística
print('c) Significância estatística dos coeficientes:')

pvalues = modelo_logit.pvalues

print('Teste de hipótese: H₀: βⱼ = 0 (variável não afeta a probabilidade)')
print(f'{"Variável":<20} {"Coef.":<12} {"P-valor":<12} {"Significância"}')
print('-' * 60)

for var in pvalues.index:
    coef = coefs_logit[var]
    pval = pvalues[var]
    if pval < 0.01:
        sig = '*** (1%)'
    elif pval < 0.05:
        sig = '** (5%)'
    elif pval < 0.10:
        sig = '* (10%)'
    else:
        sig = 'Não significativo'
    print(f'{var:<20} {coef:<12.4f} {pval:<12.4f} {sig}')

print('''
CONCLUSÃO: Todas as variáveis são estatisticamente significativas
ao nível de 1%, indicando que cada uma contribui significativamente
para explicar a probabilidade de alta renda.''')

c) Significância estatística dos coeficientes:
Teste de hipótese: H₀: βⱼ = 0 (variável não afeta a probabilidade)
Variável             Coef.        P-valor      Significância
------------------------------------------------------------
const                -9.1235      0.0000       *** (1%)
age                  0.0332       0.0000       *** (1%)
education_num        0.3543       0.0000       *** (1%)
hours_per_week       0.0311       0.0000       *** (1%)
male                 0.1233       0.0076       *** (1%)
married              2.2922       0.0000       *** (1%)
college              0.1798       0.0058       *** (1%)

CONCLUSÃO: Todas as variáveis são estatisticamente significativas
ao nível de 1%, indicando que cada uma contribui significativamente
para explicar a probabilidade de alta renda.


In [21]:
# d) Interpretação econômica dos resultados (via odds ratio)
print('d) Interpretação econômica dos resultados (Logit):\n')

# Calculando odds ratios
odds_ratios = np.exp(coefs_logit)

print('''No modelo Logit, exp(β) representa a RAZÃO DE CHANCES (Odds Ratio).
Indica quantas vezes a chance de Y=1 é multiplicada por uma mudança unitária em X.\n''')

print('ODDS RATIOS:\n')
for var in odds_ratios.index[1:]:  # Excluindo constante
    or_val = odds_ratios[var]
    pct_change = (or_val - 1) * 100
    print(f'{var}: OR = {or_val:.4f}')
    if or_val > 1:
        print(f'  -> Aumenta a chance de alta renda em {pct_change:.1f}%\n')
    else:
        print(f'  -> Diminui a chance de alta renda em {abs(pct_change):.1f}%\n')

print(f'''INTERPRETAÇÃO ECONÔMICA:

1. GÊNERO: A chance de um homem ter alta renda é {odds_ratios['male']:.2f} vezes
   a chance de uma mulher (controlando por educação, idade, etc.).
   Diferencial condicional pequeno (OR={odds_ratios['male']:.2f}); a maior parte do gap bruto
   é explicada por outras variáveis (casamento, educação, horas).

2. CASAMENTO: Casados têm chance {odds_ratios['married']:.2f}x maior de alta renda.
   Reflete o "prêmio do casamento" documentado na literatura.

3. DIPLOMA UNIVERSITÁRIO: Ter ensino superior aumenta a chance em
   {(odds_ratios['college']-1)*100:.0f}%, evidenciando forte retorno da educação.

4. EDUCAÇÃO: Cada ano adicional de estudo aumenta a chance em
   {(odds_ratios['education_num']-1)*100:.1f}% (retorno marginal da escolaridade).''')

d) Interpretação econômica dos resultados (Logit):

No modelo Logit, exp(β) representa a RAZÃO DE CHANCES (Odds Ratio).
Indica quantas vezes a chance de Y=1 é multiplicada por uma mudança unitária em X.

ODDS RATIOS:

age: OR = 1.0338
  -> Aumenta a chance de alta renda em 3.4%

education_num: OR = 1.4251
  -> Aumenta a chance de alta renda em 42.5%

hours_per_week: OR = 1.0316
  -> Aumenta a chance de alta renda em 3.2%

male: OR = 1.1312
  -> Aumenta a chance de alta renda em 13.1%

married: OR = 9.8968
  -> Aumenta a chance de alta renda em 889.7%

college: OR = 1.1969
  -> Aumenta a chance de alta renda em 19.7%

INTERPRETAÇÃO ECONÔMICA:

1. GÊNERO: A chance de um homem ter alta renda é 1.13 vezes
   a chance de uma mulher (controlando por educação, idade, etc.).
   Diferencial condicional pequeno (OR=1.13); a maior parte do gap bruto
   é explicada por outras variáveis (casamento, educação, horas).

2. CASAMENTO: Casados têm chance 9.90x maior de alta renda.
   Reflete o "prêmio d

---
## 8. Estimação do Modelo Probit

In [22]:
# a) Estimação do modelo Probit
modelo_probit = Probit(y, X).fit(disp=0)

print('a) Modelo Probit - Equação Estimada:\n')
print('P(Y=1|X) = Φ(β₀ + β₁*age + β₂*education_num + β₃*hours_per_week + β₄*male + β₅*married + β₆*college)')
print('\nOnde Φ(z) é a função de distribuição acumulada da normal padrão.\n')
print(modelo_probit.summary().tables[1])

a) Modelo Probit - Equação Estimada:

P(Y=1|X) = Φ(β₀ + β₁*age + β₂*education_num + β₃*hours_per_week + β₄*male + β₅*married + β₆*college)

Onde Φ(z) é a função de distribuição acumulada da normal padrão.

                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -5.1805      0.090    -57.853      0.000      -5.356      -5.005
age                0.0194      0.001     24.197      0.000       0.018       0.021
education_num      0.1957      0.007     26.847      0.000       0.181       0.210
hours_per_week     0.0180      0.001     21.311      0.000       0.016       0.020
male               0.0972      0.025      3.814      0.000       0.047       0.147
married            1.2809      0.023     55.582      0.000       1.236       1.326
college            0.1303      0.036      3.599      0.000       0.059       0.201


In [23]:
# b) Interpretação dos coeficientes
print('b) Interpretação dos coeficientes do Probit:\n')

coefs_probit = modelo_probit.params

print('''No modelo Probit, os coeficientes NÃO têm interpretação direta como
no MPL ou como odds ratios no Logit. Eles representam a variação no
índice latente z = Xβ, que é transformado em probabilidade via Φ(z).

O que podemos interpretar diretamente:
  - SINAL: mesma interpretação (positivo = aumenta P(Y=1))
  - MAGNITUDE RELATIVA: comparar importância entre variáveis

COEFICIENTES PROBIT:\n''')

for var in coefs_probit.index:
    print(f'{var}: {coefs_probit[var]:.4f}')

print('''\nPara interpretação quantitativa, precisamos calcular os EFEITOS
MARGINAIS (seção 10), que expressam a variação em P(Y=1) por unidade de X.''')

b) Interpretação dos coeficientes do Probit:

No modelo Probit, os coeficientes NÃO têm interpretação direta como
no MPL ou como odds ratios no Logit. Eles representam a variação no
índice latente z = Xβ, que é transformado em probabilidade via Φ(z).

O que podemos interpretar diretamente:
  - SINAL: mesma interpretação (positivo = aumenta P(Y=1))
  - MAGNITUDE RELATIVA: comparar importância entre variáveis

COEFICIENTES PROBIT:

const: -5.1805
age: 0.0194
education_num: 0.1957
hours_per_week: 0.0180
male: 0.0972
married: 1.2809
college: 0.1303

Para interpretação quantitativa, precisamos calcular os EFEITOS
MARGINAIS (seção 10), que expressam a variação em P(Y=1) por unidade de X.


In [24]:
# c) Comparação com o modelo Logit
print('c) Comparação Logit vs Probit:\n')

# Fator de escala teórico: coef_logit ≈ 1.6 * coef_probit
comparacao = pd.DataFrame({
    'Logit': coefs_logit,
    'Probit': coefs_probit,
    'Razão (L/P)': coefs_logit / coefs_probit})

print(comparacao.round(4))

razao_media = (coefs_logit / coefs_probit).mean()

print(f'''\nANÁLISE COMPARATIVA:

1. SINAIS: Idênticos em ambos os modelos (consistência qualitativa).

2. PROPORÇÃO: A razão Logit/Probit é aproximadamente {razao_media:.2f},
   entre 1.6 e 1.8 (nas variáveis bem estimadas, ~1.7-1.8).

3. SIGNIFICÂNCIA: Ambos os modelos indicam as mesmas variáveis como
   significativas, com p-valores muito similares.

4. LOG-VEROSSIMILHANÇA:
   - Logit: {modelo_logit.llf:.2f}
   - Probit: {modelo_probit.llf:.2f}
   - Valores muito próximos, indicando ajuste similar.

CONCLUSÃO: Como esperado teoricamente, os modelos produzem resultados
muito semelhantes. A escolha entre eles é geralmente baseada em
conveniência de interpretação (odds ratio no Logit) ou tradição da área.''')

c) Comparação Logit vs Probit:

                 Logit  Probit  Razão (L/P)
const          -9.1235 -5.1805       1.7611
age             0.0332  0.0194       1.7166
education_num   0.3543  0.1957       1.8100
hours_per_week  0.0311  0.0180       1.7310
male            0.1233  0.0972       1.2689
married         2.2922  1.2809       1.7896
college         0.1798  0.1303       1.3791

ANÁLISE COMPARATIVA:

1. SINAIS: Idênticos em ambos os modelos (consistência qualitativa).

2. PROPORÇÃO: A razão Logit/Probit é aproximadamente 1.64,
   entre 1.6 e 1.8 (nas variáveis bem estimadas, ~1.7-1.8).

3. SIGNIFICÂNCIA: Ambos os modelos indicam as mesmas variáveis como
   significativas, com p-valores muito similares.

4. LOG-VEROSSIMILHANÇA:
   - Logit: -11430.54
   - Probit: -11403.53
   - Valores muito próximos, indicando ajuste similar.

CONCLUSÃO: Como esperado teoricamente, os modelos produzem resultados
muito semelhantes. A escolha entre eles é geralmente baseada em
conveniência de interpret

---
## 9. Comparação dos Modelos MQO, Logit e Probit

In [25]:
# Tabela comparativa dos três modelos
print('COMPARAÇÃO DOS TRÊS MODELOS:\n')

comparacao_modelos = pd.DataFrame({
    'MPL (MQO)': modelo_mpl.params,
    'Logit': modelo_logit.params,
    'Probit': modelo_probit.params
})

print('a) Diferenças nos coeficientes:\n')
print(comparacao_modelos.round(4))

COMPARAÇÃO DOS TRÊS MODELOS:

a) Diferenças nos coeficientes:

                MPL (MQO)   Logit  Probit
const             -0.5463 -9.1235 -5.1805
age                0.0034  0.0332  0.0194
education_num      0.0345  0.3543  0.1957
hours_per_week     0.0031  0.0311  0.0180
male               0.0247  0.1233  0.0972
married            0.3087  2.2922  1.2809
college            0.1104  0.1798  0.1303


In [26]:
# b) Diferenças na significância estatística
print('b) Comparação da significância estatística (p-valores):')

pvalores_comp = pd.DataFrame({
    'MPL': modelo_mpl.pvalues,
    'Logit': modelo_logit.pvalues,
    'Probit': modelo_probit.pvalues
})

print(pvalores_comp.round(4))

print('''
OBSERVAÇÃO: Todas as variáveis são significativas a 1% nos três modelos.
Os p-valores são consistentes entre os modelos, embora ligeiramente diferentes
devido às diferentes funções de ligação e métodos de estimação.''')

b) Comparação da significância estatística (p-valores):
                  MPL  Logit  Probit
const          0.0000 0.0000  0.0000
age            0.0000 0.0000  0.0000
education_num  0.0000 0.0000  0.0000
hours_per_week 0.0000 0.0000  0.0000
male           0.0000 0.0076  0.0001
married        0.0000 0.0000  0.0000
college        0.0000 0.0058  0.0003

OBSERVAÇÃO: Todas as variáveis são significativas a 1% nos três modelos.
Os p-valores são consistentes entre os modelos, embora ligeiramente diferentes
devido às diferentes funções de ligação e métodos de estimação.


c) Vantagens e limitações de cada modelo


MODELO DE PROBABILIDADE LINEAR (MQO)
------------------------------------
Vantagens:
  - Coeficientes diretamente interpretáveis como efeitos marginais
  - Estimação simples e rápida
  - Útil para aproximação inicial

Limitações:
  - Probabilidades podem sair do intervalo [0,1]
  - Heterocedasticidade inerente
  - Efeitos marginais constantes (irrealista)

MODELO LOGIT
------------
Vantagens:
  - Probabilidades sempre entre 0 e 1
  - Interpretação via odds ratio (exp(β))
  - Caudas mais pesadas (robusto a outliers)
  - Amplamente usado em ciências sociais e saúde

Limitações:
  - Coeficientes não são efeitos marginais diretos
  - Requer cálculo adicional para efeitos marginais

MODELO PROBIT
-------------
Vantagens:
  - Probabilidades sempre entre 0 e 1
  - Fundamentação teórica via variável latente
  - Preferido em economia para modelos estruturais

Limitações:
  - Coeficientes sem interpretação direta
  - Não tem equivalente ao odds ratio
  - Caudas mais leves (sensível a outliers)
""")

---
## 10. Efeitos Marginais

In [27]:
# a) Efeitos marginais médios (Average Marginal Effects - AME)
print('a) Efeitos Marginais Médios (AME):\n')

# Calculando efeitos marginais para o modelo Logit
efeitos_marginais_logit = modelo_logit.get_margeff(at='overall')
efeitos_marginais_probit = modelo_probit.get_margeff(at='overall')

print('MODELO LOGIT - Efeitos Marginais Médios:\n')
print(efeitos_marginais_logit.summary())

a) Efeitos Marginais Médios (AME):

MODELO LOGIT - Efeitos Marginais Médios:

        Logit Marginal Effects       
Dep. Variable:          income_binary
Method:                          dydx
At:                           overall
                    dy/dx    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
age                0.0040      0.000     24.331      0.000       0.004       0.004
education_num      0.0429      0.002     27.148      0.000       0.040       0.046
hours_per_week     0.0038      0.000     21.356      0.000       0.003       0.004
male               0.0149      0.006      2.671      0.008       0.004       0.026
married            0.2777      0.004     63.340      0.000       0.269       0.286
college            0.0218      0.008      2.759      0.006       0.006       0.037


In [28]:
# b) Interpretação econômica dos efeitos marginais
print('b) Interpretação econômica dos efeitos marginais:\n')

ame = efeitos_marginais_logit.margeff
vars_names = ['age', 'education_num', 'hours_per_week', 'male', 'married', 'college']

print('''Os efeitos marginais médios representam a variação média na probabilidade
de Y=1 para uma mudança unitária em X, avaliada em todos os pontos da amostra.\n''')

for i, var in enumerate(vars_names):
    em = ame[i]
    print(f'{var}: {em:.4f}')
    if var in ['male', 'married', 'college']:
        print(f'  -> Ser {var}=1 aumenta P(renda>50K) em {em*100:.2f} pontos percentuais\n')
    else:
        print(f'  -> Aumento de 1 unidade em {var} aumenta P(renda>50K) em {em*100:.2f} p.p.\n')

print(f'''INTERPRETAÇÃO ECONÔMICA:

1. EDUCAÇÃO: Cada ano adicional de estudo aumenta a probabilidade de
   alta renda em {ame[1]*100:.2f} p.p., controlando pelos demais fatores.

2. GÊNERO: Homens têm probabilidade {ame[3]*100:.1f} p.p. maior de alta
   renda que mulheres com mesmas características observáveis.
   Nota: o gap bruto (20.0 p.p.) cai para {ame[3]*100:.1f} p.p. quando
   controlamos por casamento, educação e horas - ou seja, ~93% do gap
   bruto é explicado por essas variáveis na especificação usada.

3. CASAMENTO: Ser casado aumenta a probabilidade em {ame[4]*100:.1f} p.p.
   O "prêmio do casamento" pode refletir maior produtividade ou seleção.

4. DIPLOMA: O diploma universitário aumenta a probabilidade em
   {ame[5]*100:.1f} p.p., evidenciando forte sinalização no mercado.''')

b) Interpretação econômica dos efeitos marginais:

Os efeitos marginais médios representam a variação média na probabilidade
de Y=1 para uma mudança unitária em X, avaliada em todos os pontos da amostra.

age: 0.0040
  -> Aumento de 1 unidade em age aumenta P(renda>50K) em 0.40 p.p.

education_num: 0.0429
  -> Aumento de 1 unidade em education_num aumenta P(renda>50K) em 4.29 p.p.

hours_per_week: 0.0038
  -> Aumento de 1 unidade em hours_per_week aumenta P(renda>50K) em 0.38 p.p.

male: 0.0149
  -> Ser male=1 aumenta P(renda>50K) em 1.49 pontos percentuais

married: 0.2777
  -> Ser married=1 aumenta P(renda>50K) em 27.77 pontos percentuais

college: 0.0218
  -> Ser college=1 aumenta P(renda>50K) em 2.18 pontos percentuais

INTERPRETAÇÃO ECONÔMICA:

1. EDUCAÇÃO: Cada ano adicional de estudo aumenta a probabilidade de
   alta renda em 4.29 p.p., controlando pelos demais fatores.

2. GÊNERO: Homens têm probabilidade 1.5 p.p. maior de alta
   renda que mulheres com mesmas características 

In [29]:
# c) Comparação entre coeficientes e efeitos marginais
print('c) Comparação: Coeficientes vs Efeitos Marginais\n')

comparacao_em = pd.DataFrame({
    'Coef. MPL': modelo_mpl.params[1:].values,
    'Coef. Logit': modelo_logit.params[1:].values,
    'Coef. Probit': modelo_probit.params[1:].values,
    'EM Logit': efeitos_marginais_logit.margeff,
    'EM Probit': efeitos_marginais_probit.margeff
}, index=vars_names)

print(comparacao_em.round(4))

print('''\nOBSERVAÇÕES IMPORTANTES:

1. Os coeficientes do MPL são aproximadamente iguais aos efeitos marginais
   dos modelos Logit e Probit para as variáveis CONTÍNUAS.
   Para as DUMMIES (college, male), há divergência: o MPL impõe efeito
   constante, enquanto o Logit avalia a densidade em cada ponto.

2. Os coeficientes do Logit e Probit são MAIORES que os efeitos marginais
   porque medem o efeito no índice latente, não na probabilidade.

3. Os efeitos marginais do Logit e Probit são muito similares entre si,
   confirmando que ambos os modelos produzem conclusões equivalentes.

4. Para interpretação econômica, prefira os EFEITOS MARGINAIS, pois
   expressam diretamente a variação na probabilidade.''')

c) Comparação: Coeficientes vs Efeitos Marginais

                Coef. MPL  Coef. Logit  Coef. Probit  EM Logit  EM Probit
age                0.0034       0.0332        0.0194    0.0040     0.0041
education_num      0.0345       0.3543        0.1957    0.0429     0.0413
hours_per_week     0.0031       0.0311        0.0180    0.0038     0.0038
male               0.0247       0.1233        0.0972    0.0149     0.0205
married            0.3087       2.2922        1.2809    0.2777     0.2703
college            0.1104       0.1798        0.1303    0.0218     0.0275

OBSERVAÇÕES IMPORTANTES:

1. Os coeficientes do MPL são aproximadamente iguais aos efeitos marginais
   dos modelos Logit e Probit para as variáveis CONTÍNUAS.
   Para as DUMMIES (college, male), há divergência: o MPL impõe efeito
   constante, enquanto o Logit avalia a densidade em cada ponto.

2. Os coeficientes do Logit e Probit são MAIORES que os efeitos marginais
   porque medem o efeito no índice latente, não na probabili

---
## 11. Capacidade Preditiva do Modelo

In [30]:
# Previsões do modelo Logit
# Usando ponto de corte padrão de 0.5
prob_pred = modelo_logit.predict(X)
y_pred = (prob_pred >= 0.5).astype(int)

# a) Matriz de confusão
print('a) Matriz de Confusão:\n')

# Calculando manualmente para maior clareza
vp = ((y_pred == 1) & (y == 1)).sum()  # Verdadeiro Positivo
vn = ((y_pred == 0) & (y == 0)).sum()  # Verdadeiro Negativo
fp = ((y_pred == 1) & (y == 0)).sum()  # Falso Positivo
fn = ((y_pred == 0) & (y == 1)).sum()  # Falso Negativo

print('                    PREVISTO')
print('                  0        1')
print(f'REAL    0      {vn:5d}    {fp:5d}    (Renda <= 50K)')
print(f'        1      {fn:5d}    {vp:5d}    (Renda > 50K)')
print(f'\nTotal de observações: {len(y)}')

a) Matriz de Confusão:

                    PREVISTO
                  0        1
REAL    0      20872     1782    (Renda <= 50K)
        1       3761     3747    (Renda > 50K)

Total de observações: 30162


In [31]:
# b) Percentual de acertos (acurácia)
print('b) Percentual de Acertos (Acurácia):\n')

acuracia = (vp + vn) / len(y)
print(f'Acurácia = (VP + VN) / Total = ({vp} + {vn}) / {len(y)}')
print(f'Acurácia = {acuracia:.4f} = {acuracia*100:.2f}%')

print(f'''\nInterpretação: O modelo classifica corretamente {acuracia*100:.1f}% das
observações, o que indica boa capacidade preditiva.''')

b) Percentual de Acertos (Acurácia):

Acurácia = (VP + VN) / Total = (3747 + 20872) / 30162
Acurácia = 0.8162 = 81.62%

Interpretação: O modelo classifica corretamente 81.6% das
observações, o que indica boa capacidade preditiva.


In [32]:
# c) Sensibilidade (Taxa de Verdadeiros Positivos)
print('c) Sensibilidade (Recall / Taxa de Verdadeiros Positivos):\n')

sensibilidade = vp / (vp + fn)
print(f'Sensibilidade = VP / (VP + FN) = {vp} / ({vp} + {fn})')
print(f'Sensibilidade = {sensibilidade:.4f} = {sensibilidade*100:.2f}%')

print(f'''\nInterpretação: Entre os indivíduos que REALMENTE têm renda >50K,
o modelo identifica corretamente {sensibilidade*100:.1f}% deles.

Uma sensibilidade de {sensibilidade*100:.1f}% indica que o modelo tem
dificuldade moderada em identificar casos de alta renda.''')

c) Sensibilidade (Recall / Taxa de Verdadeiros Positivos):

Sensibilidade = VP / (VP + FN) = 3747 / (3747 + 3761)
Sensibilidade = 0.4991 = 49.91%

Interpretação: Entre os indivíduos que REALMENTE têm renda >50K,
o modelo identifica corretamente 49.9% deles.

Uma sensibilidade de 49.9% indica que o modelo tem
dificuldade moderada em identificar casos de alta renda.


In [33]:
# d) Especificidade (Taxa de Verdadeiros Negativos)
print('d) Especificidade (Taxa de Verdadeiros Negativos):')

especificidade = vn / (vn + fp)
print(f'Especificidade = VN / (VN + FP) = {vn} / ({vn} + {fp})')
print(f'Especificidade = {especificidade:.4f} = {especificidade*100:.2f}%')

print(f'''
Interpretação: Entre os indivíduos que REALMENTE têm renda <=50K,
o modelo identifica corretamente {especificidade*100:.1f}% deles.

A alta especificidade indica que o modelo é bom em identificar
casos de baixa renda, evitando falsos positivos.''')

# e) AUC-ROC - métrica adicional para avaliar o modelo
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y, prob_pred)

print(f'''
e) AUC-ROC (Area Under the ROC Curve):

AUC = {auc:.4f}

Interpretação: A AUC de {auc:.2f} indica que o modelo tem capacidade
discriminativa boa. Uma AUC de 0.5 indica modelo aleatório; 1.0 indica
discriminação perfeita. O valor obtido mostra que o modelo distingue
bem entre indivíduos de alta e baixa renda.

Nota sobre o ponto de corte: usamos 0.5, mas como a base é desbalanceada
(24% positivos), um corte menor aumentaria a sensibilidade ao custo de
mais falsos positivos.''')

print(f'''
{'='*60}
RESUMO DA CAPACIDADE PREDITIVA (Modelo Logit):
{'='*60}
Acurácia:       {acuracia*100:.2f}%
Sensibilidade:  {sensibilidade*100:.2f}%
Especificidade: {especificidade*100:.2f}%
AUC-ROC:        {auc:.4f}
{'='*60}

O modelo apresenta boa capacidade preditiva geral, com melhor
desempenho na identificação de casos de baixa renda (alta especificidade)
do que de alta renda (sensibilidade moderada). Isso é esperado dado
o desbalanceamento da amostra (maioria com renda <=50K).''')

d) Especificidade (Taxa de Verdadeiros Negativos):
Especificidade = VN / (VN + FP) = 20872 / (20872 + 1782)
Especificidade = 0.9213 = 92.13%

Interpretação: Entre os indivíduos que REALMENTE têm renda <=50K,
o modelo identifica corretamente 92.1% deles.

A alta especificidade indica que o modelo é bom em identificar
casos de baixa renda, evitando falsos positivos.

e) AUC-ROC (Area Under the ROC Curve):

AUC = 0.8701

Interpretação: A AUC de 0.87 indica que o modelo tem capacidade
discriminativa boa. Uma AUC de 0.5 indica modelo aleatório; 1.0 indica
discriminação perfeita. O valor obtido mostra que o modelo distingue
bem entre indivíduos de alta e baixa renda.

Nota sobre o ponto de corte: usamos 0.5, mas como a base é desbalanceada
(24% positivos), um corte menor aumentaria a sensibilidade ao custo de
mais falsos positivos.

RESUMO DA CAPACIDADE PREDITIVA (Modelo Logit):
Acurácia:       81.62%
Sensibilidade:  49.91%
Especificidade: 92.13%
AUC-ROC:        0.8701

O modelo apresenta bo

---
## 12. Avaliação de Afirmações (Verdadeiro ou Falso)

In [34]:
print('''
a) "Os coeficientes do modelo Logit representam diretamente a variação 
    da probabilidade."

RESPOSTA: FALSO

Justificativa: Os coeficientes do Logit representam a variação no LOG
da razão de chances (log-odds), não na probabilidade diretamente.
Para obter a variação na probabilidade, é necessário calcular os
EFEITOS MARGINAIS, que dependem do ponto de avaliação (valores de X).

Matematicamente: β = ∂log(P/(1-P))/∂X ≠ ∂P/∂X

---

b) "Os modelos Logit e Probit produzem probabilidades entre 0 e 1."

RESPOSTA: VERDADEIRO

Justificativa: Ambos os modelos utilizam funções de distribuição
acumulada (logística e normal, respectivamente) para transformar
o índice linear Xβ ∈ (-∞, +∞) em probabilidades P ∈ (0, 1).

Logit: P = exp(Xβ)/[1+exp(Xβ)] ∈ (0,1)
Probit: P = Φ(Xβ) ∈ (0,1)

---

c) "O Modelo de Probabilidade Linear pode gerar probabilidades negativas
    ou superiores a um."

RESPOSTA: VERDADEIRO

Justificativa: O MPL estima P(Y=1|X) = Xβ linearmente, sem restrições.
Para valores extremos de X, as probabilidades estimadas podem ser
< 0 ou > 1, violando a definição de probabilidade.''')

# Evidência empírica
print(f'\nEvidência empírica na nossa base:')
print(f'  - Prob. mínima estimada (MPL): {y_pred_mpl.min():.4f}')
print(f'  - Prob. máxima estimada (MPL): {y_pred_mpl.max():.4f}')


a) "Os coeficientes do modelo Logit representam diretamente a variação 
    da probabilidade."

RESPOSTA: FALSO

Justificativa: Os coeficientes do Logit representam a variação no LOG
da razão de chances (log-odds), não na probabilidade diretamente.
Para obter a variação na probabilidade, é necessário calcular os
EFEITOS MARGINAIS, que dependem do ponto de avaliação (valores de X).

Matematicamente: β = ∂log(P/(1-P))/∂X ≠ ∂P/∂X

---

b) "Os modelos Logit e Probit produzem probabilidades entre 0 e 1."

RESPOSTA: VERDADEIRO

Justificativa: Ambos os modelos utilizam funções de distribuição
acumulada (logística e normal, respectivamente) para transformar
o índice linear Xβ ∈ (-∞, +∞) em probabilidades P ∈ (0, 1).

Logit: P = exp(Xβ)/[1+exp(Xβ)] ∈ (0,1)
Probit: P = Φ(Xβ) ∈ (0,1)

---

c) "O Modelo de Probabilidade Linear pode gerar probabilidades negativas
    ou superiores a um."

RESPOSTA: VERDADEIRO

Justificativa: O MPL estima P(Y=1|X) = Xβ linearmente, sem restrições.
Para valores extr

In [35]:
print('''
d) "Os efeitos marginais dependem dos valores das variáveis explicativas."

RESPOSTA: VERDADEIRO

Justificativa: Nos modelos Logit e Probit, o efeito marginal é:

  ∂P/∂Xⱼ = f(Xβ) × βⱼ

Onde f(·) é a função densidade (derivada da FDA). Como f(Xβ) varia
com os valores de X, o efeito marginal também varia.

Exemplo: O efeito de um ano adicional de educação é MAIOR para
indivíduos com probabilidade próxima de 0.5 do que para aqueles
com probabilidade muito alta ou muito baixa.

No MPL, os efeitos marginais são CONSTANTES (= β), o que é uma
limitação do modelo.

---

e) "Logit e Probit normalmente produzem resultados bastante semelhantes."

RESPOSTA: VERDADEIRO

Justificativa: As funções logística e normal acumulada são muito
similares, diferindo principalmente nas caudas. Na prática:

  - Os sinais e significâncias são praticamente idênticos
  - Os efeitos marginais são muito próximos
  - A capacidade preditiva é equivalente
  - Os coeficientes diferem por um fator de escala (~1.6)
''')

# Evidência empírica
print('Evidência empírica - Correlação entre previsões:')
prob_logit = modelo_logit.predict(X)
prob_probit = modelo_probit.predict(X)
correlacao = np.corrcoef(prob_logit, prob_probit)[0,1]
print(f'  Correlação entre P(Y=1) do Logit e Probit: {correlacao:.6f}')
print('  (Praticamente perfeita, confirmando a semelhança dos modelos)')


d) "Os efeitos marginais dependem dos valores das variáveis explicativas."

RESPOSTA: VERDADEIRO

Justificativa: Nos modelos Logit e Probit, o efeito marginal é:

  ∂P/∂Xⱼ = f(Xβ) × βⱼ

Onde f(·) é a função densidade (derivada da FDA). Como f(Xβ) varia
com os valores de X, o efeito marginal também varia.

Exemplo: O efeito de um ano adicional de educação é MAIOR para
indivíduos com probabilidade próxima de 0.5 do que para aqueles
com probabilidade muito alta ou muito baixa.

No MPL, os efeitos marginais são CONSTANTES (= β), o que é uma
limitação do modelo.

---

e) "Logit e Probit normalmente produzem resultados bastante semelhantes."

RESPOSTA: VERDADEIRO

Justificativa: As funções logística e normal acumulada são muito
similares, diferindo principalmente nas caudas. Na prática:

  - Os sinais e significâncias são praticamente idênticos
  - Os efeitos marginais são muito próximos
  - A capacidade preditiva é equivalente
  - Os coeficientes diferem por um fator de escala (~1.6)

Evid